# Assignment 4
In this week's assignment you will fit and evaluate multiple classification models based on the Heart.csv dataset from ISL, predicting whether the subjects have a heart disease (AHD). There are two main learning goals:
- Gain more practical experience with writing analysis code, this time focusing specifically on reusing code through loops and functions.
- Compare approaches for model performance evaluation.
The tasks are organized such that you should be able to reuse code from the early tasks in the subsequent ones.

### 0. Prepare the dataset for the subsequent modelling.

#### 0.2 Load the dataset and drop all variables except the predictors Age, Sex, ChestPain, RestBP, Chol, and the target variable AHD. Drop all rows containing a NaN value.

In [22]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import sklearn

heart = pd.read_csv('Heart.csv')
columns_to_keep = ['Age', 'Sex', 'ChestPain', 'RestBP', 'Chol', 'AHD']
heart = heart.drop(columns=[col for col in heart.columns if col not in columns_to_keep]).dropna()

#Using a for-loop to remove all the columns not specified in the variable columns_to_keep
#.dropna() removes rows with NaN-values. 
#Verifying by checking the dataframe heart

heart.head()

,Age,Sex,ChestPain,RestBP,Chol,AHD
0,63,1,typical,145,233,No
1,67,1,asymptomatic,160,286,Yes
2,67,1,asymptomatic,120,229,Yes
3,37,1,nonanginal,130,250,No
4,41,0,nontypical,130,204,No


#### 0.3 Onehot encode the variable ChestPain. This means that where you before had a single column with one of four values ['typical', 'asymptomatic', 'nonanginal', 'nontypical'], you will now have four binary columns (their names don't matter), akin to 'ChestPain_typical' 'ChestPain_asymptomatic', 'ChestPain_nonanginal', 'ChestPain_nontypical'. 
#### A row that before had ChestPain='typical' will now have ChestPain_typical=1 and the other three columns set to 0, ChestPain='asymptomatic' will have ChestPain_asymptomatic=1 and the other three set to 0, etc.

In [24]:
#Using the pandas function to dummy code chest pain into four different true/false columns
heart = pd.get_dummies(heart, columns=['ChestPain'], prefix='ChestPain')

#It returned true/false values, so here we identify the four columns that start with chestpain, and replace the values with 1 and 0 for true and false
chest_pain_columns = [col for col in heart.columns if col.startswith('ChestPain_')]
heart[chest_pain_columns] = heart[chest_pain_columns].astype(int)

#Inspecting the dataframe to see the effects
print(heart.head())

   Age  Sex  RestBP  Chol  AHD  ChestPain_asymptomatic  ChestPain_nonanginal  \
0   63    1     145   233   No                       0                     0   
1   67    1     160   286  Yes                       1                     0   
2   67    1     120   229  Yes                       1                     0   
3   37    1     130   250   No                       0                     1   
4   41    0     130   204   No                       0                     0   

   ChestPain_nontypical  ChestPain_typical  
0                     0                  1  
1                     0                  0  
2                     0                  0  
3                     0                  0  
4                     1                  0  


#### 0.4 Binary encode the target variable AHD such that 'No'=0 and 'Yes'=1.

In [26]:
heart['AHD'] = (heart['AHD'] == 'Yes').astype('int8')
print(heart.head())

   Age  Sex  RestBP  Chol  AHD  ChestPain_asymptomatic  ChestPain_nonanginal  \
0   63    1     145   233    0                       0                     0   
1   67    1     160   286    1                       1                     0   
2   67    1     120   229    1                       1                     0   
3   37    1     130   250    0                       0                     1   
4   41    0     130   204    0                       0                     0   

   ChestPain_nontypical  ChestPain_typical  
0                     0                  1  
1                     0                  0  
2                     0                  0  
3                     0                  0  
4                     1                  0  


### 1. Fit a model using a standard train/validation split through multiple steps.
Through the steps you will practice chaining functions, and you will also create the infrastructure necessary for the remaining tasks.

#### 1.1 Write a function "stratified_split" that takes three arguments: A dataframe, a number of folds, and a list of variables to stratify by. 
#### The function should return a list of dataframes, one for each fold, where the dataframes are stratified by the variables in the list. 
#### Test that the function works by splitting the dataset into two folds based on 'AHD', 'Age' and 'RestBP' and 
#### print the size of each fold, 
#### print the counts of 0s and 1s in AHD, and 
#### print the mean of each of 'Age' and 'RestBP' (all these should be printed individually per fold). 
#### Ensure that the function does not modify the original dataframe.

In [29]:
from sklearn.model_selection import StratifiedKFold

#Using general terms so the code can be applied to other datasets.
#This should split the dataframe (df) into stratified folds (n_folds) using specific variables (stratify_vars)
#Copying the original dataframe so we don't modify it. Generic name again so it works on other sets.
def stratified_split(df, n_folds, stratify_vars):
    df_copy = df.copy()

    strata = []
    for var in stratify_vars:
        if pd.api.types.is_numeric_dtype(df_copy[var]):
            strata.append(pd.qcut(df_copy[var], q=5, duplicates='drop', labels=False).astype(str))
        else:
            strata.append(df_copy[var].astype(str))

    df_copy['strata'] = ["_".join(items) for items in zip(*strata)]

#Initializing stratified kfold
    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42) 

#Creating the folds
    folds = []
    for _, test_index in skf.split(df_copy, df_copy['strata']):
        fold = df_copy.iloc[test_index].drop(columns=['strata']).copy()
        folds.append(fold)
    return folds 

#Calling folds
folds = stratified_split(heart, n_folds=2, stratify_vars=['AHD', 'Age', 'RestBP'])

#Printing the folds to verify
for i, fold in enumerate(folds):
    print(f"\nFold {i+1}:")
    print(f"Size: {len(fold)} rows")
    print("AHD value counts:")
    print(fold['AHD'].value_counts())
    print(f"Mean Age: {fold['Age'].mean():.1f}")
    print(f"Mean RestBP: {fold['RestBP'].mean():.1f}")

print(heart.head())
print("\n\n", str(fold['Age'].head()))



Fold 1:
Size: 152 rows
AHD value counts:
AHD
0    85
1    67
Name: count, dtype: int64
Mean Age: 54.7
Mean RestBP: 130.9

Fold 2:
Size: 151 rows
AHD value counts:
AHD
0    79
1    72
Name: count, dtype: int64
Mean Age: 54.2
Mean RestBP: 132.5
   Age  Sex  RestBP  Chol  AHD  ChestPain_asymptomatic  ChestPain_nonanginal  \
0   63    1     145   233    0                       0                     0   
1   67    1     160   286    1                       1                     0   
2   67    1     120   229    1                       1                     0   
3   37    1     130   250    0                       0                     1   
4   41    0     130   204    0                       0                     0   

   ChestPain_nontypical  ChestPain_typical  
0                     0                  1  
1                     0                  0  
2                     0                  0  
3                     0                  0  
4                     1                  0  


 1 

#### 1.2 Write a function 'fit_and_predict' that takes 4 arguments: 
#### A training set, 
#### a validation set, 
#### a list of predictors, and 
#### a target variable. 
#### The function should fit a logistic regression model to the training set using the predictors and target variable, and return the predictions of the model on the validation set.

In [31]:
from sklearn.linear_model import LogisticRegression

#I'm adding explanations of the functions from now on so they can be applied more easily to other projects

#Function: fit_and_predict
#This function fits a logistic regression model to the training set and returns predictions on the validation set

#Parameters used:
#train_set (pd.DataFrame) - Training dataset
#val_set (pd.DataFrame) - Validation dataset
#predictors (list) - List of column names to use as predictors
#target (str) - Column name of the target variable

#It returns:
#An array of predictions on the validation set - np.ndarray

def fit_and_predict(train_set, val_set, predictors, target):    
# Step 1: Defining the model
    model = LogisticRegression(max_iter=1000)    
# Step 2: Fitting the model on the training set
    X_train = train_set[predictors] 
    y_train = train_set[target]
    model.fit(X_train, y_train)
# Step 3: Make predictions on the validation set
    X_val = val_set[predictors]
    predictions = model.predict(X_val)
    return predictions

#testing the function on the heart dataset
predictors = ['Age', 'RestBP', 'Chol']
target = 'AHD'

#Using the folds from before
train_set = folds[0]
val_set = folds[1]

#Drumroll... calling predictions
predictions = fit_and_predict(train_set, val_set, predictors, target)

# Print predictions
print(predictions)

#But does it work, is it effective? Let's have a look at the balanced accuracy
from sklearn.metrics import balanced_accuracy_score
# Calculating balanced accuracy
balanced_accuracy = balanced_accuracy_score(val_set[target], predictions)
print(f"Balanced Accuracy: {balanced_accuracy:.2f}")

#The predictors and target can be switched with any of the ones in the dataset.

[1 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 1 1 1 0 0 0 0 0 1 0 1 0 0 0 1 1 0 1 1 0
 0 0 1 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0 1 0 0 1 0 1 0 0 0 1 0 1 0 0 1 0 0 0
 1 0 0 0 1 0 0 0 0 1 0 1 1 0 0 0 0 0 1 0 1 1 1 0 1 1 1 0 1 0 0 0 0 0 0 0 0
 0 0 1 0 0 1 0 0 0 0 1 1 1 0 1 1 1 1 1 0 0 1 0 1 1 0 1 0 0 0 0 0 1 0 0 1 1
 0 0 1]
Balanced Accuracy: 0.56


#### 1.3 Write a function 'fit_and_predict_standardized' that takes 5 arguments: 
#### A training set, 
#### a validation set, 
#### a list of predictors, 
#### a target variable, 
#### and a list of variables to standardize.

#### Using a loop (or a scaler), the function should z-score standardize the given variables in both the training set and the validation set based on the mean and standard deviation in the training set. 

#### Then, the function should call the 'fit_and_predict' function and return its result. Ensure that the function does not modify the original dataframes. Test the function using the train and validation set from above (e.g. the two folds from the split), while standardizing the 'Age', 'RestBP' and 'Chol' variables (as mentioned above, the target should be AHD, and you should also include the remaining predictors: 'Sex' and the ChestPain-variables)

In [33]:
#Function: fit_and_predict_standardized
#This function standardizes specified variables ('Age', 'RestBP', 'Chol') in both training and validation sets, 
#trains a logistic regression model and makes predictions

#Parameters used:
#train_set (pd.DataFrame) - Training dataset
#val_set (pd.DataFrame) - Validation dataset
#predictors (list) - List of column names to use as predictors
#target (str) - Column name of the target variable
#vars_to_standardize (list) - List of variables to z-score standardize

#It returns
# An array of predictions on the validation set - np.ndarray

def fit_and_predict_standardized(train_set, val_set, predictors, target, vars_to_standardize):
# Step 1: Making copies of the dataframes to avoid modifying originals
    train_copy = train_set.copy()
    val_copy = val_set.copy()
# Step 2: Standardizing specified variables
    for var in vars_to_standardize:
# Computing mean and standard deviation from training set
        mean = train_copy[var].mean()
        std = train_copy[var].std()
# Standardizing in training set
        train_copy[var] = (train_copy[var] - mean) / std
# Standardizing in validation set using training set stats
        val_copy[var] = (val_copy[var] - mean) / std
# Step 3: Calling fit_and_predict using standardized data
    predictions = fit_and_predict(train_copy, val_copy, predictors, target)
    return predictions

#Testing the function

predictors = ['Age', 'RestBP', 'Chol', 'Sex', 
              'ChestPain_asymptomatic', 'ChestPain_nonanginal', 
              'ChestPain_nontypical', 'ChestPain_typical']
target = 'AHD'
vars_to_standardize = ['Age', 'RestBP', 'Chol']

#Using the folds from 1.2 for training and validation 
train_set = folds[0]
val_set = folds[1]

# Calling the fit_and_predict_standardized function
predictions = fit_and_predict_standardized(train_set, val_set, predictors, target, vars_to_standardize)

# Printing predictions
print(predictions)

# Trying the balanced accuracy for good measure
balanced_accuracy = balanced_accuracy_score(val_set[target], predictions)
print(f"Balanced Accuracy: {balanced_accuracy:.2f}")

#Not bad, it performs better than the unstandardized prediction model.

[1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 1 0 0 0 1 0 1 0 0 1 1 1 0 0 1 1 0 0 1 0 0
 1 0 1 0 0 0 0 1 1 0 0 1 0 0 1 1 1 0 0 1 1 1 1 0 0 0 1 0 0 0 0 0 0 1 0 0 1
 1 1 1 0 1 0 0 0 0 1 1 1 1 1 1 0 1 0 1 0 1 1 1 1 1 1 1 0 0 1 1 1 1 0 0 0 0
 0 0 0 0 1 0 0 1 0 0 0 0 1 1 0 1 0 0 1 1 1 1 1 1 0 0 0 1 0 0 0 1 1 0 0 0 1
 0 0 1]
Balanced Accuracy: 0.79


#### 1.4 Write a function 'fit_and_compute_auc' that takes 5 arguments: A training set, a validation set, a list of predictors, a target variable, and a list of variables to standardize. The function should call the 'fit_and_predict_standardized' function to retrieve out-of-sample predictions for the validation set. Based on these and the ground truth labels in the validation set, it should compute and return the AUC. Test the function using the train and test set from above, while standardizing the 'Age', 'RestBP' and 'Chol' variables (and including the remaining predictors). Print the AUC.

In [35]:
from sklearn.metrics import roc_auc_score

#Function: fit_and_compute_auc
# This function standardizes specified variables ('Age', 'RestBP', 'Chol'), trains a logistic regression model and 
#computes the AUC on the validation set
#It uses the following parameters:

#train_set (pd.DataFrame) - Training dataset
#val_set (pd.DataFrame) - Validation dataset
#predictors (list) - List of column names to use as predictors
#target (str) - Column name of the target variable
#vars_to_standardize (list) - List of variables to z-score standardize

#It returns a float, the AUC score for the validation set.


def fit_and_compute_auc(train_set, val_set, predictors, target, vars_to_standardize):
    # Step 1: Get predictions using fit_and_predict_standardized
    predictions = fit_and_predict_standardized(train_set, val_set, predictors, target, vars_to_standardize)
    
    # Step 2: Compute AUC using ground truth labels and predicted probabilities
    auc = roc_auc_score(val_set[target], predictions)
    
    return auc

predictors = ['Age', 'RestBP', 'Chol', 'Sex', 
              'ChestPain_asymptomatic', 'ChestPain_nonanginal', 
              'ChestPain_nontypical', 'ChestPain_typical']
target = 'AHD'
vars_to_standardize = ['Age', 'RestBP', 'Chol']

train_set = folds[0]  # Fold 1 as training set
val_set = folds[1]    # Fold 2 as validation set

# Calling the fit_and_compute_auc function
auc = fit_and_compute_auc(train_set, val_set, predictors, target, vars_to_standardize)

# Printing AUC
print(f"AUC: {auc:.2f}")

AUC: 0.79


### 2. Perform a cross-validation.
Use the 'stratified_split' function to split the dataset into 10 folds, stratified on variables you find reasonable. For each fold, use the 'fit_and_compute_auc' function to compute the AUC of the model on the held-out validation set. Print the mean and standard deviation of the AUCs across the 10 folds.

In [64]:
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np

def stratified_split(df, n_folds, stratify_col='AHD'):
    """Custom stratified split function that maintains class balance"""
    df = df.copy()
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    return [df.iloc[val_idx] for _, val_idx in skf.split(df, df[stratify_col])]

# Load your heart dataset here
# heart = pd.read_csv(...)

# Define variables
predictors = ['Age', 'RestBP', 'Chol', 'Sex', 
              'ChestPain_asymptomatic', 'ChestPain_nonanginal',
              'ChestPain_nontypical', 'ChestPain_typical']
target = 'AHD'
vars_to_standardize = ['Age', 'RestBP', 'Chol']

# Create 10 stratified folds
folds = stratified_split(heart, n_folds=10)

# Cross-validation
auc_scores = []
for i in range(10):
    train_set = pd.concat([folds[j] for j in range(10) if j != i])
    val_set = folds[i]
    
    auc = fit_and_compute_auc(
        train_set=train_set,
        val_set=val_set,
        predictors=predictors,
        target=target,
        vars_to_standardize=vars_to_standardize
    )
    auc_scores.append(auc)

print(f"Mean AUC: {np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}")


Mean AUC: 0.784 ± 0.072


### OPTIONAL 3. Use the bootstrap to achieve a distribution of out-of-bag AUCs.

### 4. Theory

#### 4.1 List some benefits of wrapping code in functions rather than copying and pasting it multiple times.

There is less chance of typing/copying something wrong when you reuse a function rather than an entire block of code. It can also be good to make the functions general so you can apply them to different datasets in different settings. If you add an explanation for what the function does it's also easier to share it with others, and you could maybe make a library of similar functions, for example for preprocessing or analysis of data. If you need to make changes to the function for some reason you can do it in one place rather than fixing the code in all the places where it's implemented. 

#### 4.2 Explain three classification metrics and their benefits and drawbacks.

A classification metric evaluates how accurate the predictions of our models are compared to the true values of our verification set. We get to see the variance/bias tradeoff here. We want a model that is quite accurate, but not overfitted, because a model that is too biased will not fare well with the variance of the outside world. There are many ways to test accuracy, and they all have pros and cons. It's good practice to use several metrics so we get a better picture of the accuracy.

The confusion matrix is used for binary classification, and is easily interpretable. It returns how many true positives, false positives, true negatives and false negatives we had in the validation set. The problem with the confusion matrix is that it's not good if we have imbalanced classes, like in the heart dataset. In data where one class is much more common than the other (like "no ahd"), the model will get a good score if it simply guesses that no one has a disease. It's not sensitive enough to pick up accuracy when there are class imbalances.

Accuracy is another interpretable classification metric. It is defined as the true positive and the true negative over the true positive, true negative, false positive and false negative. Like the confusion matrix it does not account for the cost of misclassification (returning "no heart disease" if you actually have heart disease, for instance), and it does not handle imbalanced classes well. Instead it is often better to use the balanced accuracy measure, as it averages sensitivity for both classes. 

Instead of (or rather in addition to) these two measures, we can use area under the curve, as in exercise 1.4. An AUC-score will range from 0.5-1, with 0.5 being guesswork and 1 being a perfect score. The benefits of the AUC is that it handles imbalanced classes well because it measures performance across all classification thresholds, and it focuses on whether predictions are ranked correctly. Since you can choose your thresholds with this measure, you can make the model more sensitive so it will return more false positives than true negatives (which is what we would want if we were screening for diseases with a high mortality rate).

#### 4.3 Write a couple of sentences comparing the three methods (train/validation, cross-validation, bootstrap) above as approaches to quantify model performance. Which one yielded the best results? Which one would you expect to yield the best results? Can you mention some theoretical benefits and drawbacks with each? Even if you didn't do the optional bootstrap exercise you should reflect on this as an approach.

The train/validation split we have used splits the dataset into a training set (80%) and a validation set (20%). We simply test the model we have trained on the validation set and see how accurate the predictions are. This method uses only a subset of the data for training our model. Problems can occur if the dataset is split unevenly with respect to qualities that matter for our predictions (like if all the heart disease cases ended up in the validation set instead of the training set, the model would predict that everyone is healthy). This method does not return a confidence interval.

Cross-validation has the advantage that we use the entire dataset to train our models. Unlike the train/validation split we get several estimates of the error, and a mean value to combine them. We used K-fold cross-validation, where we split the data into k number of folds. We then train the different folds and test it on the validation set. It is computationally more expensive than a regular train/validation split, but gives us a more reliable performance estimate. It is good for imbalanced classes if we use stratification first. Drawbacks can be that the results still depend on how we split the dataset, and that we make several models instead of one, so we can no longer use parameter estimates and p-values. 

With the bootstrap method we draw random samples from the population with replacement, so the same sample can be drawn several times. This method uses the entire dataset for training. It can add more power to our analysis because we can potentially draw endless samples to create new datasets (pulling ourselves up by the bootstraps), and evaluate performance across them. We can get a confidence interval with bootstrapping because the repeated resampling creates an empirical distribution of our statistic from which we can directly calculate percentile-based intervals.

If I could choose only one of these three methods for our dataset (and computational power was not a problem), I think I would go for the stratified k-fold because it would probably give more reliable predictions with imbalanced classes.

#### 4.4 Why do we stratify the dataset before splitting?

For the same reason we want representative samples in the first place. What we really want to do with these statistical methods is to say something about the general population, we use our samples as models for the real world. When we stratify our dataset we make sure that each training set and validation set are roughly the same when it comes to certain aspects we think might be important for the outcome/predictions.

#### 4.5 What other use cases can you think of for the bootstrap method?

We can use it for smaller datasets to "boost" our collection.